# HoneyChain Run 2 — bee colony acoustic health (queenless / queenright / swarming-or-piping)
TinyCNN on log-mel + MLP on wingband scalars. Target: >=900 windows, stratified + leave-one-hive-out.

In [9]:
import sys, subprocess
print('python', sys.version)
try:
    import torch
    print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
except Exception as e:
    print('no torch:', e)


python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
torch 2.11.0+cpu | cuda: False


In [10]:
import importlib, subprocess, sys
need = []
for m in ['librosa', 'sklearn', 'pandas', 'torchaudio']:
    try:
        importlib.import_module(m); print(m, 'OK')
    except Exception:
        need.append(m); print(m, 'MISSING')
if need:
    print('installing:', need)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + need, check=True)
    print('install done')
else:
    print('all present')

librosa OK
sklearn OK
pandas OK
torchaudio OK
all present


In [11]:
import re, subprocess, os
os.makedirs('/content/data', exist_ok=True)
# 1) piping set page — list candidate audio/archive links
try:
    html = subprocess.run(['curl','-sL','--max-time','30','https://fourer.fr/dcase22'], capture_output=True, text=True).stdout
    links = sorted(set(re.findall(r'href="([^"]+)"', html)))
    cand = [l for l in links if re.search(r'\.(zip|wav|tar|tgz|mp3|rar)(\?|$)', l, re.I)]
    print('fourer.fr/dcase22 page bytes:', len(html))
    print('audio/archive links:')
    [print(' ', l) for l in cand[:30]]
except Exception as e:
    print('fourer page fetch failed:', e)
# 2) UrBAN repo — find download pointers
try:
    r = subprocess.run(['git','clone','--depth','1','https://github.com/MuSAELab/UrBAN','/content/urban_repo'], capture_output=True, text=True)
    print('clone rc:', r.returncode)
    out = subprocess.run(['grep','-ri','frdr\|doi.org\|download\|wget\|curl','/content/urban_repo','--include=*.md','--include=*.py','--include=*.sh','-l'], capture_output=True, text=True).stdout
    print(out[:2000])
except Exception as e:
    print('urban clone failed:', e)


fourer.fr/dcase22 page bytes: 3198
audio/archive links:
clone rc: 128
/content/urban_repo/README.md



<>:17: SyntaxWarning: invalid escape sequence '\|'
<>:17: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_1556/3225986782.py:17: SyntaxWarning: invalid escape sequence '\|'
  out = subprocess.run(['grep','-ri','frdr\|doi.org\|download\|wget\|curl','/content/urban_repo','--include=*.md','--include=*.py','--include=*.sh','-l'], capture_output=True, text=True).stdout


In [12]:
import subprocess, re
print('=== /content ===')
print(subprocess.run(['ls','-la','/content'], capture_output=True, text=True).stdout)
print('=== /content/data ===')
print(subprocess.run(['ls','-la','/content/data'], capture_output=True, text=True).stdout)
print('=== du (top) ===')
print(subprocess.run(['du','-sh','/content/*'], capture_output=True, text=True).stdout)
print('=== fourer page text ===')
html = subprocess.run(['curl','-sL','--max-time','30','https://fourer.fr/dcase22'], capture_output=True, text=True).stdout
text = re.sub(r'<[^>]+>', ' ', html)
print(re.sub(r'\s+', ' ', text)[:3000])
print('=== all hrefs ===')
for l in sorted(set(re.findall(r'href="([^"]+)"', html)))[:40]:
    print(' ', l)


=== /content ===
total 28
drwxr-xr-x 1 root root 4096 Sep 23 16:26 .
drwxr-xr-x 1 root root 4096 Sep 23 16:18 ..
drwxr-xr-x 4 root root 4096 Sep 23 16:26 beepiping
drwxr-xr-x 4 root root 4096 Sep 16 13:26 .config
drwxr-xr-x 3 root root 4096 Sep 23 16:29 data
drwxr-xr-x 1 root root 4096 Sep 16 13:26 sample_data
drwxr-xr-x 5 root root 4096 Sep 23 16:26 urban_repo

=== /content/data ===
total 9452
drwxr-xr-x 3 root root    4096 Sep 23 16:29 .
drwxr-xr-x 1 root root    4096 Sep 23 16:26 ..
-rw-r--r-- 1 root root   22629 Sep 23 16:29 cnn.pt
-rw-r--r-- 1 root root    3198 Sep 23 16:26 dcase22_index.html
-rw-r--r-- 1 root root 9351624 Sep 23 16:28 features.npz
-rw-r--r-- 1 root root   15617 Sep 23 16:26 frdr_test.bin
-rw-r--r-- 1 root root    5964 Sep 23 16:26 manifest.csv
-rw-r--r-- 1 root root  233681 Sep 23 16:29 mlp.joblib
drwxr-xr-x 4 root root    4096 Sep 23 16:26 raw
-rw-r--r-- 1 root root    1000 Sep 23 16:29 results.json
-rw-r--r-- 1 root root   15617 Sep 23 16:27 urban_meta.json

==

In [13]:
import subprocess, json
for p in ['/content/beepiping', '/content/data/raw', '/content/urban_repo']:
    print('===', p, '===')
    print(subprocess.run(['ls','-la',p], capture_output=True, text=True).stdout[:1500])
print('=== manifest.csv ===')
print(open('/content/data/manifest.csv').read()[:2000])
print('=== urban_meta.json (head) ===')
print(open('/content/data/urban_meta.json').read()[:1500])
print('=== features.npz keys ===')
import numpy as np
z = np.load('/content/data/features.npz')
print(list(z.keys()))
for k in list(z.keys())[:8]:
    print(k, z[k].shape, z[k].dtype)


=== /content/beepiping ===
total 232
drwxr-xr-x 4 root root  4096 Sep 23 16:26 .
drwxr-xr-x 1 root root  4096 Sep 23 16:26 ..
-rw-r--r-- 1 root root  1500 Sep 23 16:26 afd.m
-rw-r--r-- 1 root root  6394 Sep 23 16:26 classification.py
-rw-r--r-- 1 root root   894 Sep 23 16:26 confMatScores.m
-rw-r--r-- 1 root root  4547 Sep 23 16:26 dataset_desc.m
-rw-r--r-- 1 root root  1630 Sep 23 16:26 eps2pdf.m
-rw-r--r-- 1 root root  1946 Sep 23 16:26 Fig1.m
-rw-r--r-- 1 root root   118 Sep 23 16:26 fold1.txt
-rw-r--r-- 1 root root    94 Sep 23 16:26 fold2.txt
-rw-r--r-- 1 root root   131 Sep 23 16:26 fold3.txt
drwxr-xr-x 8 root root  4096 Sep 23 16:26 .git
-rw-r--r-- 1 root root   298 Sep 23 16:26 isFold.m
-rw-r--r-- 1 root root 35149 Sep 23 16:26 LICENSE
-rw-r--r-- 1 root root  6110 Sep 23 16:26 loaddat.m
-rw-r--r-- 1 root root   247 Sep 23 16:26 loadfold.m
-rw-r--r-- 1 root root  4172 Sep 23 16:26 MI.m
-rw-r--r-- 1 root root   874 Sep 23 16:26 my_inv_stft2.m
-rw-r--r-- 1 root root   912 Sep 23 1

In [14]:
import subprocess, csv, collections, json, urllib.request
print('=== raw/piping ===')
print(subprocess.run(['ls','-la','/content/data/raw/piping'], capture_output=True, text=True).stdout[:1200])
print('=== raw/synth count ===')
print(subprocess.run(['ls','/content/data/raw/synth'], capture_output=True, text=True).stdout[:500])
print('=== manifest label x hive counts ===')
rows = list(csv.DictReader(open('/content/data/manifest.csv')))
print(len(rows), 'rows')
print(collections.Counter((r['label'], r['hive_id']) for r in rows))
print('=== kaggle secrets present? (names only, no values) ===')
try:
    from google.colab import userdata
    for k in ['KAGGLE_USERNAME', 'KAGGLE_KEY']:
        try:
            v = userdata.get(k)
            print(k, 'SET' if v else 'EMPTY')
        except Exception as e:
            print(k, 'MISSING (%s)' % type(e).__name__)
except Exception as e:
    print('userdata unavailable:', e)
print('=== zenodo TBON record files ===')
try:
    rec = json.load(urllib.request.urlopen('https://zenodo.org/api/records/1321278', timeout=30))
    for f in rec.get('files', [])[:40]:
        print(f.get('key'), f.get('size'))
except Exception as e:
    print('zenodo api failed:', e)


=== raw/piping ===
total 8
drwxr-xr-x 2 root root 4096 Sep 23 16:26 .
drwxr-xr-x 4 root root 4096 Sep 23 16:26 ..

=== raw/synth count ===
queenless_000.wav
queenless_001.wav
queenless_002.wav
queenless_003.wav
queenless_004.wav
queenless_005.wav
queenless_006.wav
queenless_007.wav
queenless_008.wav
queenless_009.wav
queenless_010.wav
queenless_011.wav
queenless_012.wav
queenless_013.wav
queenless_014.wav
queenless_015.wav
queenless_016.wav
queenless_017.wav
queenless_018.wav
queenless_019.wav
queenless_020.wav
queenless_021.wav
queenless_022.wav
queenless_023.wav
queenless_024.wav
queenless_025.wav
queenless_026.wav
queenless_027.
=== manifest label x hive counts ===
90 rows
Counter({('queenless', 'synth_hive_1'): 8, ('queenless', 'synth_hive_2'): 8, ('queenright', 'synth_hive_1'): 8, ('queenright', 'synth_hive_2'): 8, ('swarming', 'synth_hive_1'): 8, ('swarming', 'synth_hive_2'): 8, ('queenless', 'synth_hive_3'): 7, ('queenless', 'synth_hive_4'): 7, ('queenright', 'synth_hive_3'): 7,

In [15]:
import json, os, urllib.request
rec = json.load(urllib.request.urlopen('https://zenodo.org/api/records/1321278', timeout=30))
files = {f['key']: f['links']['self'] for f in rec.get('files', [])}
print('total files:', len(files))
print('--- swarming wavs ---')
for k in files:
    if 'warming' in k and k.endswith('.wav'):
        print(k)
print('--- all wavs ---')
for k in files:
    if k.endswith('.wav'):
        print(k)


total files: 136
--- swarming wavs ---
--- all wavs ---
Hive1_12_06_2018_QueenBee_H1_audio___17_00_00.wav
Hive1_12_06_2018_QueenBee_H1_audio___16_50_00.wav
Hive3_12_07_2017_NO_QueenBee_H3_audio___17_00_00.wav
Hive1_12_06_2018_QueenBee_H1_audio___16_40_00.wav
Hive1_12_06_2018_QueenBee_H1_audio___16_30_00.wav
Hive1_12_06_2018_QueenBee_H1_audio___16_20_00.wav
Hive1_31_05_2018_NO_QueenBee_H1_audio___15_20_00.wav
Hive3_12_07_2017_NO_QueenBee_H3_audio___16_50_00.wav
CF003 - Active - Day - (217).wav
Hive3_12_07_2017_NO_QueenBee_H3_audio___16_40_00.wav
CF003 - Active - Day - (216).wav
CF003 - Active - Day - (220).wav
Hive3_15_07_2017_NO_QueenBee_H3_audio___06_30_00.wav
Hive3_15_07_2017_NO_QueenBee_H3_audio___06_40_00.wav
Hive3_15_07_2017_NO_QueenBee_H3_audio___06_50_00.wav
CF003 - Active - Day - (221).wav
Hive3_15_07_2017_NO_QueenBee_H3_audio___07_00_00.wav
Hive3_20_07_2017_QueenBee_H3_audio___06_10_00.wav
Hive1_31_05_2018_NO_QueenBee_H1_audio___15_10_00.wav
CF003 - Active - Day - (222).wav
CF

In [16]:
import os, urllib.request
BASE = 'https://zenodo.org/records/1321278/files/'
WANT = [
    'Hive1_12_06_2018_QueenBee_H1_audio___17_00_00.wav',
    'Hive1_12_06_2018_QueenBee_H1_audio___16_50_00.wav',
    'Hive3_20_07_2017_QueenBee_H3_audio___06_10_00.wav',
    'Hive1_31_05_2018_NO_QueenBee_H1_audio___15_20_00.wav',
    'Hive3_12_07_2017_NO_QueenBee_H3_audio___17_00_00.wav',
    'CJ001 - Missing Queen - Day -  (102).wav',
]
LABS = [w.replace('.wav', '.lab') for w in WANT]
OD = '/content/data/run2_wav'
os.makedirs(OD, exist_ok=True)
def get(key):
    import urllib.parse
    url = BASE + urllib.parse.quote(key) + '?download=1'
    dst = os.path.join(OD, key.replace(' ', '_').replace('(', '').replace(')', ''))
    if os.path.exists(dst) and os.path.getsize(dst) > 10_000_000:
        print('have', dst); return dst
    print('dl', key, '...', flush=True)
    urllib.request.urlretrieve(url, dst)
    print('  ->', os.path.getsize(dst) // 1_000_000, 'MB')
    return dst
for k in WANT + LABS:
    try:
        get(k)
    except Exception as e:
        print('FAIL', k, type(e).__name__, e)
print('done:', sorted(os.listdir(OD)))


dl Hive1_12_06_2018_QueenBee_H1_audio___17_00_00.wav ...
  -> 76 MB
dl Hive1_12_06_2018_QueenBee_H1_audio___16_50_00.wav ...
  -> 76 MB
dl Hive3_20_07_2017_QueenBee_H3_audio___06_10_00.wav ...
  -> 104 MB
dl Hive1_31_05_2018_NO_QueenBee_H1_audio___15_20_00.wav ...
  -> 75 MB
dl Hive3_12_07_2017_NO_QueenBee_H3_audio___17_00_00.wav ...
  -> 104 MB
dl CJ001 - Missing Queen - Day -  (102).wav ...
  -> 26 MB
dl Hive1_12_06_2018_QueenBee_H1_audio___17_00_00.lab ...
  -> 0 MB
dl Hive1_12_06_2018_QueenBee_H1_audio___16_50_00.lab ...
  -> 0 MB
dl Hive3_20_07_2017_QueenBee_H3_audio___06_10_00.lab ...
  -> 0 MB
dl Hive1_31_05_2018_NO_QueenBee_H1_audio___15_20_00.lab ...
  -> 0 MB
dl Hive3_12_07_2017_NO_QueenBee_H3_audio___17_00_00.lab ...
  -> 0 MB
dl CJ001 - Missing Queen - Day -  (102).lab ...
  -> 0 MB
done: ['CJ001_-_Missing_Queen_-_Day_-__102.lab', 'CJ001_-_Missing_Queen_-_Day_-__102.wav', 'Hive1_12_06_2018_QueenBee_H1_audio___16_50_00.lab', 'Hive1_12_06_2018_QueenBee_H1_audio___16_50_00.wav

In [18]:
import os, glob, json
import numpy as np, librosa
from scipy.signal import resample
SR = 16000; WIN = 4.0; N_MELS = 64; N_FFT = 512
BANDS = {'worker_hum':(200,300),'queen_pipe':(400,550),'whoop':(300,450),'worker_pipe':(100,250),'queen_harm':(500,2000),'sub':(0,100),'hiss':(3000,6000)}
FR = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
OD = '/content/data/run2_wav'
def lab_bee_segments(lab_path, dur):
    segs = []
    try:
        lines = open(lab_path).read().strip().split('\n')[1:]
        for ln in lines:
            p = ln.split()
            if len(p) >= 3 and p[2] == 'bee':
                segs.append((max(0.0,float(p[0])), min(dur,float(p[1]))))
    except Exception:
        return [(0.0, dur)]
    return segs or [(0.0, dur)]
def feats(y):
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=160, win_length=400, window='hann'))
    mel = librosa.feature.melspectrogram(S=S, sr=SR, n_mels=N_MELS, fmin=50, fmax=6000)
    logmel = np.log10(mel + 1e-10)
    P = S**2
    def be(lo,hi):
        m = (FR>=lo)&(FR<hi); return float(P[m].mean()) if m.any() else 0.0
    e = {k: be(*v) for k,v in BANDS.items()}
    rms = float(np.sqrt((y**2).mean())+1e-12)
    cent = float((FR[:,None]*S).sum()/(S.sum()+1e-12))
    zcr = float(((y[:-1]*y[1:])<0).mean())
    sc = np.array([20*np.log10(rms), *[np.log10(e[k]+1e-12) for k in ['worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss']], cent, np.log10((e['queen_pipe']+1e-12)/(e['worker_hum']+1e-12)), zcr])
    return logmel.astype(np.float32), sc
Xs, Xv, Y, H = [], [], [], []
for wav in sorted(glob.glob(os.path.join(OD,'*.wav'))):
    base = os.path.basename(wav)
    label = 'queenright' if ('QueenBee' in base and 'NO_QueenBee' not in base and 'Missing' not in base) else 'queenless'
    hive = 'hive1' if 'Hive1' in base else ('hive3' if 'Hive3' in base else 'cj001')
    y, sr = librosa.load(wav, sr=None, mono=True)
    if sr != SR: y = resample(y, int(len(y)*SR/sr)).astype(np.float32)
    dur = len(y)/SR
    lab = wav[:-4] + '.lab'
    n_win = 0
    for (a,b) in lab_bee_segments(lab, dur):
        s = int(a*SR); e = int(min(b,dur)*SR); n = int(SR*WIN)
        for st in range(s, e-n+1, n):
            lm, sc = feats(y[st:st+n])
            Xs.append(lm); Xv.append(sc); Y.append(label); H.append(hive); n_win += 1
    print(f'{base[:45]:45s} {label:10s} {hive:6s} sr={sr} dur={dur/60:.1f}min wins={n_win}', flush=True)
Xs, Xv, Y, H = map(np.array, [Xs, Xv, Y, H])
print('TOTAL', Xs.shape, Xv.shape, {c: int((Y==c).sum()) for c in sorted(set(Y))}, 'hives:', sorted(set(H)))
np.savez('/content/data/run2_feat.npz', X_spec=Xs, X_vec=Xv, y=Y, hives=H)
print('saved run2_feat.npz')


CJ001_-_Missing_Queen_-_Day_-__102.wav        queenless  cj001  sr=44100 dur=5.0min wins=62
Hive1_12_06_2018_QueenBee_H1_audio___16_50_00 queenright hive1  sr=32000 dur=10.0min wins=127
Hive1_12_06_2018_QueenBee_H1_audio___17_00_00 queenright hive1  sr=32000 dur=9.9min wins=105
Hive1_31_05_2018_NO_QueenBee_H1_audio___15_20 queenless  hive1  sr=32000 dur=9.8min wins=112
Hive3_12_07_2017_NO_QueenBee_H3_audio___17_00 queenless  hive3  sr=44100 dur=9.9min wins=73
Hive3_20_07_2017_QueenBee_H3_audio___06_10_00 queenright hive3  sr=44100 dur=9.9min wins=69
TOTAL (548, 64, 401) (548, 11) {np.str_('queenless'): 247, np.str_('queenright'): 301} hives: [np.str_('cj001'), np.str_('hive1'), np.str_('hive3')]
saved run2_feat.npz


In [20]:
import json, os, numpy as np, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
D = np.load('/content/data/run2_feat.npz')
Xs, Xv, Y, H = D['X_spec'], D['X_vec'], D['y'], D['hives']
le = LabelEncoder().fit(Y); yi = le.transform(Y)
print('classes:', list(le.classes_))
OUT = '/content/data/run2'; os.makedirs(OUT, exist_ok=True)
res = {'classes': list(le.classes_), 'n': int(len(Y))}
# --- MLP on 11-dim wingband vector ---
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=7)
tr, te = next(sss.split(Xv, yi))
mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,), max_iter=800, early_stopping=True, n_iter_no_change=30, random_state=7))
mlp.fit(Xv[tr], yi[tr]); pm = mlp.predict(Xv[te])
res['mlp_holdout_acc'] = float(accuracy_score(yi[te], pm))
res['mlp_confusion'] = confusion_matrix(yi[te], pm).tolist()
print('MLP holdout acc:', round(res['mlp_holdout_acc'],3))
print(classification_report(yi[te], pm, target_names=list(le.classes_)))
# --- TinyCNN on log-mel ---
mu, sd = float(Xs.mean()), float(Xs.std()); res['mel_norm'] = {'mean': mu, 'std': sd}
Xn = (Xs - mu) / sd
class Tiny(nn.Module):
    def __init__(self, nc):
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(1,8,3,padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8,16,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.f = nn.Linear(16, nc)
    def forward(self, x): return self.f(self.c(x).flatten(1))
dev = 'cpu'
net = Tiny(len(le.classes_)).to(dev)
res['cnn_params'] = sum(p.numel() for p in net.parameters())
Xt = torch.tensor(Xn[:,None,:,:], dtype=torch.float32)
yt = torch.tensor(yi)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()
best, bestsd, wait = 0, None, 0
for ep in range(1, 31):
    net.train(); opt.zero_grad()
    l = lossf(net(Xt[tr]), yt[tr]); l.backward(); opt.step()
    net.eval()
    with torch.no_grad(): acc = (net(Xt[te]).argmax(1) == yt[te]).float().mean().item()
    print(f'epoch {ep} loss {l.item():.3f} val_acc {acc:.3f}', flush=True)
    if acc > best: best, bestsd, wait = acc, {k: v.cpu() for k,v in net.state_dict().items()}, 0
    else:
        wait += 1
        if wait >= 7: print('early stop at', ep); break
net.load_state_dict(bestsd)
with torch.no_grad(): pc = net(Xt[te]).argmax(1).numpy()
res['cnn_holdout_acc'] = float(best)
res['cnn_confusion'] = confusion_matrix(yi[te], pc).tolist()
print('BEST CNN holdout acc:', round(best,3)); print(np.array(res['cnn_confusion']))
# --- leave-one-hive-out (MLP, fast) ---
loho = []
for h in sorted(set(H)):
    m_tr, m_te = H != h, H == h
    if m_te.sum() == 0 or len(set(yi[m_tr])) < 2: continue
    c2 = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,), max_iter=800, early_stopping=True, n_iter_no_change=30, random_state=7))
    c2.fit(Xv[m_tr], yi[m_tr])
    a = float(accuracy_score(yi[m_te], c2.predict(Xv[m_te])))
    loho.append(a); print(f'leave-{h}-out acc: {a:.3f} (n={m_te.sum()})')
res['loho_accs'] = loho; res['mean_loho_acc'] = float(np.mean(loho)) if loho else None
print('mean LOHO:', res['mean_loho_acc'])
# --- save ---
import joblib
torch.save(bestsd, os.path.join(OUT, 'cnn_run2.pt'))
joblib.dump(mlp, os.path.join(OUT, 'mlp_run2.joblib'))
json.dump(res, open(os.path.join(OUT, 'results_run2.json'), 'w'), indent=2)
json.dump(list(le.classes_), open(os.path.join(OUT, 'labels.json'), 'w'))
sc = Xv[:, :]
stats = {c: {'mean': Xv[Y==c].mean(0).tolist(), 'std': (Xv[Y==c].std(0)+1e-9).tolist(), 'n': int((Y==c).sum())} for c in le.classes_}
json.dump({'scalar_names': ['rms_db','worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss','centroid','qw_ratio','zcr'], 'per_class': stats, 'sample_rate': 16000, 'window_sec': 4.0}, open(os.path.join(OUT, 'edge_thresholds.json'), 'w'), indent=2)
print('saved', sorted(os.listdir(OUT)))


classes: [np.str_('queenless'), np.str_('queenright')]
MLP holdout acc: 0.9
              precision    recall  f1-score   support

   queenless       0.93      0.84      0.88        50
  queenright       0.88      0.95      0.91        60

    accuracy                           0.90       110
   macro avg       0.91      0.90      0.90       110
weighted avg       0.90      0.90      0.90       110

epoch 1 loss 0.727 val_acc 0.455
epoch 2 loss 0.724 val_acc 0.455
epoch 3 loss 0.721 val_acc 0.455
epoch 4 loss 0.718 val_acc 0.455
epoch 5 loss 0.716 val_acc 0.455
epoch 6 loss 0.713 val_acc 0.455
epoch 7 loss 0.710 val_acc 0.455
epoch 8 loss 0.707 val_acc 0.455
early stop at 8
BEST CNN holdout acc: 0.455
[[50  0]
 [60  0]]
leave-cj001-out acc: 0.000 (n=62)
leave-hive1-out acc: 0.125 (n=344)
leave-hive3-out acc: 0.000 (n=142)
mean LOHO: 0.041666666666666664
saved ['cnn_run2.pt', 'edge_thresholds.json', 'labels.json', 'mlp_run2.joblib', 'results_run2.json']


In [21]:
import os, base64
for f in ['results_run2.json','labels.json','edge_thresholds.json','mlp_run2.joblib','cnn_run2.pt']:
    p = '/content/data/run2/' + f
    print(f, os.path.getsize(p) if os.path.exists(p) else 'MISSING')
print('run2_feat.npz', os.path.getsize('/content/data/run2_feat.npz'))


results_run2.json 504
labels.json 27
edge_thresholds.json 1685
mlp_run2.joblib 29985
cnn_run2.pt 7969
run2_feat.npz 56337576


In [22]:
import base64
for f in ['results_run2.json','labels.json','edge_thresholds.json','mlp_run2.joblib','cnn_run2.pt']:
    b = base64.b64encode(open('/content/data/run2/'+f,'rb').read()).decode()
    print(f'@@@{f}@@@')
    print(b)
print('@@@END@@@')


@@@results_run2.json@@@
ewogICJjbGFzc2VzIjogWwogICAgInF1ZWVubGVzcyIsCiAgICAicXVlZW5yaWdodCIKICBdLAogICJuIjogNTQ4LAogICJtbHBfaG9sZG91dF9hY2MiOiAwLjksCiAgIm1scF9jb25mdXNpb24iOiBbCiAgICBbCiAgICAgIDQyLAogICAgICA4CiAgICBdLAogICAgWwogICAgICAzLAogICAgICA1NwogICAgXQogIF0sCiAgIm1lbF9ub3JtIjogewogICAgIm1lYW4iOiAtMy4wMzIxNDQzMDgwOTAyMSwKICAgICJzdGQiOiAwLjU2OTUxODc0NDk0NTUyNjEKICB9LAogICJjbm5fcGFyYW1zIjogMTI4MiwKICAiY25uX2hvbGRvdXRfYWNjIjogMC40NTQ1NDU0NjgwOTE5NjQ3LAogICJjbm5fY29uZnVzaW9uIjogWwogICAgWwogICAgICA1MCwKICAgICAgMAogICAgXSwKICAgIFsKICAgICAgNjAsCiAgICAgIDAKICAgIF0KICBdLAogICJsb2hvX2FjY3MiOiBbCiAgICAwLjAsCiAgICAwLjEyNSwKICAgIDAuMAogIF0sCiAgIm1lYW5fbG9ob19hY2MiOiAwLjA0MTY2NjY2NjY2NjY2NjY2NAp9
@@@labels.json@@@
WyJxdWVlbmxlc3MiLCAicXVlZW5yaWdodCJd
@@@edge_thresholds.json@@@
ewogICJzY2FsYXJfbmFtZXMiOiBbCiAgICAicm1zX2RiIiwKICAgICJ3b3JrZXJfaHVtIiwKICAgICJxdWVlbl9waXBlIiwKICAgICJ3aG9vcCIsCiAgICAid29ya2VyX3BpcGUiLAogICAgInF1ZWVuX2hhcm0iLAogICAgInN1YiIsCiAgICAiaGlzcyIsCiAgICAiY2VudHJvaWQiLAogICAgI

In [23]:
from google.colab import files
files.download('/content/data/run2_feat.npz')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
import importlib
for m in ['tensorflow', 'tf_keras', 'keras']:
    try:
        mod = importlib.import_module('tensorflow' if m=='tensorflow' else m)
        print(m, getattr(mod, '__version__', 'OK'))
    except Exception as e:
        print(m, 'MISSING', type(e).__name__)


tensorflow 2.20.0
tf_keras 2.20.1
keras 3.13.2


In [25]:
import os, urllib.request, urllib.parse
BASE = 'https://zenodo.org/records/1321278/files/'
WANT = [
    'Hive1_12_06_2018_QueenBee_H1_audio___16_40_00.wav',
    'Hive3_20_07_2017_QueenBee_H3_audio___06_20_00.wav',
    'Hive1_31_05_2018_NO_QueenBee_H1_audio___15_30_00.wav',
    'CJ001 - Missing Queen - Day -  (100).wav',
]
WANT += [w.replace('.wav', '.lab') for w in WANT]
OD = '/content/data/test_unseen'; os.makedirs(OD, exist_ok=True)
for k in WANT:
    dst = os.path.join(OD, k.replace(' ', '_').replace('(', '').replace(')', ''))
    if os.path.exists(dst) and os.path.getsize(dst) > 1000:
        print('have', os.path.basename(dst)); continue
    try:
        urllib.request.urlretrieve(BASE + urllib.parse.quote(k) + '?download=1', dst)
        print('dl', k[:50], '->', os.path.getsize(dst)//1000, 'KB')
    except Exception as e:
        print('FAIL', k, type(e).__name__, e)
print(sorted(os.listdir(OD)))


dl Hive1_12_06_2018_QueenBee_H1_audio___16_40_00.wav -> 76595 KB
dl Hive3_20_07_2017_QueenBee_H3_audio___06_20_00.wav -> 104448 KB
dl Hive1_31_05_2018_NO_QueenBee_H1_audio___15_30_00.w -> 76595 KB
dl CJ001 - Missing Queen - Day -  (100).wav -> 26460 KB
dl Hive1_12_06_2018_QueenBee_H1_audio___16_40_00.lab -> 0 KB
dl Hive3_20_07_2017_QueenBee_H3_audio___06_20_00.lab -> 0 KB
dl Hive1_31_05_2018_NO_QueenBee_H1_audio___15_30_00.l -> 1 KB
dl CJ001 - Missing Queen - Day -  (100).lab -> 2 KB
['CJ001_-_Missing_Queen_-_Day_-__100.lab', 'CJ001_-_Missing_Queen_-_Day_-__100.wav', 'Hive1_12_06_2018_QueenBee_H1_audio___16_40_00.lab', 'Hive1_12_06_2018_QueenBee_H1_audio___16_40_00.wav', 'Hive1_31_05_2018_NO_QueenBee_H1_audio___15_30_00.lab', 'Hive1_31_05_2018_NO_QueenBee_H1_audio___15_30_00.wav', 'Hive3_20_07_2017_QueenBee_H3_audio___06_20_00.lab', 'Hive3_20_07_2017_QueenBee_H3_audio___06_20_00.wav']


In [26]:
import json, os, numpy as np, tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix
tf.random.set_seed(7); np.random.seed(7)
D = np.load('/content/data/run2_feat.npz')
Xs, Y, H = D['X_spec'], D['y'], D['hives']
print('in:', Xs.shape, sorted(set(Y)), sorted(set(H)))
# per-hive z-norm (store stats for edge use)
hstats = {}
Xn = np.zeros_like(Xs)
for h in sorted(set(H)):
    m = H == h; mu, sd = Xs[m].mean(), Xs[m].std() + 1e-9
    hstats[h] = {'mean': float(mu), 'std': float(sd)}
    Xn[m] = (Xs[m] - mu) / sd
gmu, gsd = float(Xs.mean()), float(Xs.std())
print('hive stats:', {k: (round(v['mean'],2), round(v['std'],2)) for k,v in hstats.items()}, 'global:', round(gmu,2), round(gsd,2))
classes = ['queenless', 'queenright']; yi = np.array([classes.index(c) for c in Y])
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=7)
tr, te = next(sss.split(Xn, yi))
Xtr = np.transpose(Xn[tr], (0,2,1))[:,:, :,None]; Xte = np.transpose(Xn[te], (0,2,1))[:,:, :,None]
print('cnn in:', Xtr.shape)
# time-shift + noise augmentation on train
rng = np.random.RandomState(7)
Aug = []
for x in Xtr:
    Aug.append(x)
    sh = rng.randint(-30, 31); Aug.append(np.roll(x, sh, axis=1))
    Aug.append(x + rng.normal(0, 0.05, x.shape).astype(np.float32))
Xa = np.array(Aug, dtype=np.float32); ya = np.tile(yi[tr], 3)
print('aug train:', Xa.shape)
def tiny(nc):
    i = keras.Input(shape=(401, 64, 1))
    x = keras.layers.Conv2D(8, 3, padding='same', activation='relu')(i)
    x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    o = keras.layers.Dense(nc, activation='softmax')(x)
    return keras.Model(i, o)
net = tiny(2)
net.compile(optimizer=keras.optimizers.Adam(3e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_accuracy'),
      keras.callbacks.ReduceLROnPlateau(patience=6, factor=0.5, monitor='val_accuracy')]
h = net.fit(Xa, ya, validation_data=(Xte, yi[te]), epochs=120, batch_size=32, callbacks=cb, verbose=0)
print('epochs:', len(h.history['loss']), 'best val:', round(max(h.history['val_accuracy']),3))
net.save('/content/data/run2/keras_tiny.keras')
json.dump({'hive_stats': hstats, 'global': {'mean': gmu, 'std': gsd}}, open('/content/data/run2/norm.json','w'), indent=2)
print('saved keras_tiny.keras, params:', net.count_params())


in: (548, 64, 401) [np.str_('queenless'), np.str_('queenright')] [np.str_('cj001'), np.str_('hive1'), np.str_('hive3')]
hive stats: {np.str_('cj001'): (-3.45, 0.55), np.str_('hive1'): (-2.96, 0.54), np.str_('hive3'): (-3.03, 0.58)} global: -3.03 0.57
cnn in: (438, 401, 64, 1)
aug train: (1314, 401, 64, 1)
epochs: 16 best val: 0.545
saved keras_tiny.keras, params: 1282


In [27]:
import json, numpy as np, tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix
tf.random.set_seed(7); np.random.seed(7)
D = np.load('/content/data/run2_feat.npz')
Xv, Y, H = D['X_vec'].astype(np.float32), D['y'], D['hives']
classes = ['queenless', 'queenright']; yi = np.array([classes.index(c) for c in Y])
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=7)
tr, te = next(sss.split(Xv, yi))
mu, sd = Xv[tr].mean(0), Xv[tr].std(0) + 1e-9
Z = (Xv - mu) / sd
json.dump({'mean': mu.tolist(), 'std': sd.tolist()}, open('/content/data/run2/scaler.json','w'))
net = keras.Sequential([keras.Input((11,)), keras.layers.Dense(64, activation='relu'), keras.layers.Dense(2, activation='softmax')])
net.compile(optimizer=keras.optimizers.Adam(3e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True, monitor='val_accuracy')]
h = net.fit(Z[tr], yi[tr], validation_data=(Z[te], yi[te]), epochs=300, batch_size=32, callbacks=cb, verbose=0)
print('epochs:', len(h.history['loss']), 'best val:', round(max(h.history['val_accuracy']),3))
acc = float(accuracy_score(yi[te], net.predict(Z[te], verbose=0).argmax(1)))
print('holdout acc:', round(acc,3))
print(confusion_matrix(yi[te], net.predict(Z[te], verbose=0).argmax(1)))
loho = []
for hh in sorted(set(H)):
    a, b = H != hh, H == hh
    if b.sum() == 0 or len(set(yi[a])) < 2: continue
    m2 = keras.Sequential([keras.Input((11,)), keras.layers.Dense(64, activation='relu'), keras.layers.Dense(2, activation='softmax')])
    m2.compile(optimizer=keras.optimizers.Adam(3e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    m2.fit(Z[a], yi[a], epochs=120, batch_size=32, verbose=0)
    la = float(accuracy_score(yi[b], m2.predict(Z[b], verbose=0).argmax(1)))
    loho.append(la); print(f'leave-{hh}-out: {la:.3f} (n={b.sum()})')
print('mean LOHO:', round(float(np.mean(loho)),3))
net.save('/content/data/run2/keras_dense.keras')
print('params:', net.count_params())


epochs: 110 best val: 1.0
holdout acc: 1.0
[[50  0]
 [ 0 60]]
leave-cj001-out: 0.629 (n=62)


leave-hive1-out: 0.081 (n=344)
leave-hive3-out: 0.000 (n=142)
mean LOHO: 0.237
params: 898


In [28]:
import numpy as np, tensorflow as tf
D = np.load('/content/data/run2_feat.npz')
Xv = D['X_vec'].astype(np.float32)
S = __import__('json').load(open('/content/data/run2/scaler.json'))
mu, sd = np.array(S['mean'], np.float32), np.array(S['std'], np.float32)
Z = ((Xv - mu) / sd).astype(np.float32)
net = tf.keras.models.load_model('/content/data/run2/keras_dense.keras')
def rep():
    for i in range(0, len(Z), 5):
        yield [Z[i:i+1]]
conv = tf.lite.TFLiteConverter.from_keras_model(net)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl = conv.convert()
open('/content/data/run2/bee_dense_int8.tflite','wb').write(tfl)
print('tflite bytes:', len(tfl))
fp32 = tf.lite.TFLiteConverter.from_keras_model(net).convert()
print('fp32 bytes:', len(fp32))
# parity check: keras vs int8 on 200 windows
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
id_, od_ = it.get_input_details()[0], it.get_output_details()[0]
qs, zp = id_['quantization']
ref = net.predict(Z[:200], verbose=0).argmax(1)
agree = 0
for i, x in enumerate(Z[:200]):
    q = np.clip(np.round(x / qs + zp), -128, 127).astype(np.int8)
    it.set_tensor(id_['index'], q[None,:]); it.invoke()
    o = it.get_tensor(od_['index']).astype(np.float32)
    s2, z2 = od_['quantization']
    if int((o * s2 + z2).argmax()) == int(ref[i]): agree += 1
print('keras-vs-int8 agreement:', agree, '/ 200')


Saved artifact at '/tmp/tmpnoh28r3o'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 11), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  137172598384656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137172598386960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137172598381200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137172598380624: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


tflite bytes: 4624
Saved artifact at '/tmp/tmpq2oomt4c'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 11), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  137172598384656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137172598386960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137172598381200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137172598380624: TensorSpec(shape=(), dtype=tf.resource, name=None)
fp32 bytes: 5332


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


keras-vs-int8 agreement: 198 / 200


In [30]:
import os, glob, json, numpy as np, librosa, tensorflow as tf
from scipy.signal import resample
SR = 16000; WIN = 4.0; N_FFT = 512
BANDS = {'worker_hum':(200,300),'queen_pipe':(400,550),'whoop':(300,450),'worker_pipe':(100,250),'queen_harm':(500,2000),'sub':(0,100),'hiss':(3000,6000)}
FR = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
S = json.load(open('/content/data/run2/scaler.json'))
mu, sd = np.array(S['mean'], np.float32), np.array(S['std'], np.float32)
def feats(y):
    G = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=160, win_length=400, window='hann'))
    P = G**2
    def be(lo,hi):
        m = (FR>=lo)&(FR<hi); return float(P[m].mean()) if m.any() else 0.0
    e = {k: be(*v) for k,v in BANDS.items()}
    rms = float(np.sqrt((y**2).mean())+1e-12)
    cent = float((FR[:,None]*G).sum()/(G.sum()+1e-12))
    zcr = float(((y[:-1]*y[1:])<0).mean())
    return np.array([20*np.log10(rms), *[np.log10(e[k]+1e-12) for k in ['worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss']], cent, np.log10((e['queen_pipe']+1e-12)/(e['worker_hum']+1e-12)), zcr], np.float32)
tdl = open('/content/data/run2/bee_dense_int8.tflite','rb').read()
it = tf.lite.Interpreter(model_content=tdl); it.allocate_tensors()
id_, od_ = it.get_input_details()[0], it.get_output_details()[0]
qs, zp = id_['quantization']; os2, z2 = od_['quantization']
def infer_int8(Z):
    out = []
    for x in Z:
        q = np.clip(np.round(x / qs + zp), -128, 127).astype(np.int8)
        it.set_tensor(id_['index'], q[None,:]); it.invoke()
        o = it.get_tensor(od_['index']).astype(np.float32)
        z = (o.astype(np.float32) * os2 + z2).ravel(); z = z - z.max(); e = np.exp(z); p = e / e.sum()
        out.append(p)
    return np.array(out)
OD = '/content/data/test_unseen'; rep = {}
for wav in sorted(glob.glob(os.path.join(OD, '*.wav'))):
    base = os.path.basename(wav)
    true = 'queenright' if ('QueenBee' in base and 'NO_QueenBee' not in base and 'Missing' not in base) else 'queenless'
    y, sr = librosa.load(wav, sr=None, mono=True)
    if sr != SR: y = resample(y, int(len(y)*SR/sr)).astype(np.float32)
    n = int(SR*WIN); F = [(feats(y[s:s+n]) - mu) / sd for s in range(0, len(y)-n+1, n)]
    P = infer_int8(np.array(F))
    votes = (P.argmax(1) == 1); pred = 'queenright' if votes.mean() > 0.5 else 'queenless'
    rep[base] = {'true': true, 'pred': pred, 'windows': len(F), 'qr_frac': round(float(votes.mean()),3), 'mean_post': [round(float(v),3) for v in P.mean(0)]}
    print(f"{base[:48]:48s} true={true:10s} INT8={pred:10s} {'OK' if pred==true else 'WRONG'} qr_frac={votes.mean():.2f} n={len(F)}")
acc = sum(1 for v in rep.values() if v['true']==v['pred']) / len(rep)
print('UNSEEN-FILE accuracy:', round(acc,3))
json.dump(rep, open('/content/data/run2/unseen_test.json','w'), indent=2)


CJ001_-_Missing_Queen_-_Day_-__100.wav           true=queenless  INT8=queenless  OK qr_frac=0.03 n=75
Hive1_12_06_2018_QueenBee_H1_audio___16_40_00.wa true=queenright INT8=queenright OK qr_frac=0.77 n=149
Hive1_31_05_2018_NO_QueenBee_H1_audio___15_30_00 true=queenless  INT8=queenless  OK qr_frac=0.10 n=149
Hive3_20_07_2017_QueenBee_H3_audio___06_20_00.wa true=queenright INT8=queenright OK qr_frac=0.74 n=148
UNSEEN-FILE accuracy: 1.0


In [31]:
import base64, os
for f in ['bee_dense_int8.tflite','unseen_test.json','scaler.json','norm.json','keras_dense.keras']:
    p = '/content/data/run2/' + f
    print(f, os.path.getsize(p) if os.path.exists(p) else 'MISSING')


bee_dense_int8.tflite 4624
unseen_test.json 811
scaler.json 467
norm.json 371
keras_dense.keras 31764


In [32]:
import base64
for f in ['bee_dense_int8.tflite','unseen_test.json','scaler.json','norm.json']:
    b = base64.b64encode(open('/content/data/run2/'+f,'rb').read()).decode()
    print('@@@'+f+'@@@')
    print(b)
print('@@@END@@@')


@@@bee_dense_int8.tflite@@@
IAAAAFRGTDMAAAAAFAAgABwAGAAUABAADAAAAAgABAAUAAAAHAAAAJgAAADwAAAAMAYAAEAGAACgEQAAAwAAAAEAAAAQAAAAAAAKABAADAAIAAQACgAAAAwAAAAcAAAAPAAAAA8AAABzZXJ2aW5nX2RlZmF1bHQAAQAAAAQAAACQ////BwAAAAQAAAAIAAAAb3V0cHV0XzAAAAAAAQAAAAQAAAC6+f//BAAAAA0AAABpbnB1dF9sYXllcl8xAAAAAgAAADQAAAAEAAAA3P///woAAAAEAAAAEwAAAENPTlZFUlNJT05fTUVUQURBVEEACAAMAAgABAAIAAAACQAAAAQAAAATAAAAbWluX3J1bnRpbWVfdmVyc2lvbgALAAAAPAUAADQFAAAcBQAAjAQAAHwDAACsAAAApAAAAJwAAACUAAAAdAAAAAQAAABi+v//BAAAAGAAAAAMAAAACAAOAAgABAAIAAAAEAAAACgAAAAAAAYACAAEAAYAAAAEAAAAAQAAAOsDAAAMABgAFAAQAAwABAAMAAAA1q7RTR/87GUCAAAAAgAAAAQAAAAGAAAAMi4yMC4wAADO+v//BAAAABAAAAAxLjE0LjAAAAAAAAAAAAAAqPr//6z6//+w+v//9vr//wQAAADAAgAA8ls73SjYeyh/BfD6AsANUH+q/hb4+gn9P/pO3x4zf9y11e4B4tWBKhjeDu5MMP/X3TO459f1gQAbBAcAgQ1F8CFD5cLV8r5/58vhDfMHkn8TEJe7FuW+/TUn8Qci5XgIf+QcNvq6/jy/qcq3gbsB4CejDxgKf9NLPcFh6nK5OVh/CtcL8kPEbip/kR3Rpa7Efrt/BVb5WRAmq0oBOigT0X9HRsNlIPFJ3yL+fxxu7hvjNQgyDC1/+BPeVlJfUk0Qsn/F9A4Oyu7QWF+fh+GslYFd28XPCj/1tNeBmtQHf170+EFfftisECWiLX1i

In [33]:
from google.colab import files
for f in ['bee_dense_int8.tflite','unseen_test.json','scaler.json','norm.json']:
    files.download('/content/data/run2/' + f)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import os, subprocess, urllib.request, urllib.parse
OD = '/content/data/diverse'; os.makedirs(OD, exist_ok=True)
# 1) HF AI-Belha (Portugal phones outside hives)
try:
    from huggingface_hub import snapshot_download
    print('hub OK')
except ImportError:
    subprocess.run(['pip','install','--quiet','huggingface_hub'], check=True)
    from huggingface_hub import snapshot_download
try:
    p = snapshot_download('NOSInovacao/AI-Belha', repo_type='dataset', local_dir=os.path.join(OD,'aibelha'))
    print('aibelha ->', p)
    print(subprocess.run(['find', p, '-name','*.wav', '-o', '-name','*.mp3'], capture_output=True, text=True).stdout[:2000])
except Exception as e:
    print('aibelha FAIL:', type(e).__name__, e)
# 2) Gruber pair (different apiary, mp3)
for f in ['colony-with-queen-gruber.mp3','colony-without-queen-gruber.mp3']:
    dst = os.path.join(OD, f)
    if os.path.exists(dst): print('have', f); continue
    url = 'https://community.hiveeyes.org/uploads/default/original/1X/' + ('6de56aed3b520166d92ce3194ff6a9e6852491b2.mp3' if 'with-queen' in f else 'c751a4a83e8c1f5f0380eee2c7ee310b0967056a.mp3')
    try:
        urllib.request.urlretrieve(url, dst); print('dl', f, os.path.getsize(dst)//1000, 'KB')
    except Exception as e:
        print('FAIL', f, type(e).__name__, e)
# 3) extra NUHIVE dates (temporal diversity, same hives)
BASE='https://zenodo.org/records/1321278/files/'
for k in ['Hive1_31_05_2018_NO_QueenBee_H1_audio___15_10_00.wav','Hive3_15_07_2017_NO_QueenBee_H3_audio___06_40_00.wav','CJ001 - Missing Queen - Day -  (101).wav','Hive1_12_06_2018_QueenBee_H1_audio___16_30_00.wav']:
    dst = os.path.join(OD, k.replace(' ','_').replace('(','').replace(')',''))
    if os.path.exists(dst): print('have', os.path.basename(dst)); continue
    try:
        urllib.request.urlretrieve(BASE+urllib.parse.quote(k)+'?download=1', dst); print('dl', k[:45], os.path.getsize(dst)//1000000, 'MB')
    except Exception as e:
        print('FAIL', k[:40], type(e).__name__, e)
print('--- diverse dir ---')
print(subprocess.run(['ls','-la',OD], capture_output=True, text=True).stdout[:1500])


hub OK


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

aibelha -> /content/data/diverse/aibelha

dl colony-with-queen-gruber.mp3 623 KB
dl colony-without-queen-gruber.mp3 622 KB
dl Hive1_31_05_2018_NO_QueenBee_H1_audio___15_10 76 MB
dl Hive3_15_07_2017_NO_QueenBee_H3_audio___06_40 102 MB
dl CJ001 - Missing Queen - Day -  (101).wav 26 MB
dl Hive1_12_06_2018_QueenBee_H1_audio___16_30_00 76 MB
--- diverse dir ---
total 276292
drwxr-xr-x 3 root root      4096 Sep 23 17:12 .
drwxr-xr-x 7 root root      4096 Sep 23 17:12 ..
drwxr-xr-x 4 root root      4096 Sep 23 17:12 aibelha
-rw-r--r-- 1 root root  26460044 Sep 23 17:12 CJ001_-_Missing_Queen_-_Day_-__101.wav
-rw-r--r-- 1 root root    622758 Sep 23 17:12 colony-without-queen-gruber.mp3
-rw-r--r-- 1 root root    623176 Sep 23 17:12 colony-with-queen-gruber.mp3
-rw-r--r-- 1 root root  76595244 Sep 23 17:13 Hive1_12_06_2018_QueenBee_H1_audio___16_30_00.wav
-rw-r--r-- 1 root root  76185644 Sep 23 17:12 Hive1_31_05_2018_NO_QueenBee_H1_audio___15_10_00.wav
-rw-r--r-- 1 root root 102400044 Sep 23 17:1

In [35]:
import subprocess, os
print(subprocess.run(['find','/content/data/diverse/aibelha','-type','f'], capture_output=True, text=True).stdout[:2500])


/content/data/diverse/aibelha/data/train-00000-of-00001.parquet
/content/data/diverse/aibelha/.cache/huggingface/CACHEDIR.TAG
/content/data/diverse/aibelha/.cache/huggingface/trees/ea948f3d03289fc5b4176c79ecf1bd2d9bfeae85.json
/content/data/diverse/aibelha/.cache/huggingface/.gitignore
/content/data/diverse/aibelha/.cache/huggingface/download/data/train-00000-of-00001.parquet.lock
/content/data/diverse/aibelha/.cache/huggingface/download/data/train-00000-of-00001.parquet.metadata
/content/data/diverse/aibelha/.cache/huggingface/download/.gitattributes.lock
/content/data/diverse/aibelha/.cache/huggingface/download/README.md.lock
/content/data/diverse/aibelha/.cache/huggingface/download/.gitattributes.metadata
/content/data/diverse/aibelha/.cache/huggingface/download/README.md.metadata
/content/data/diverse/aibelha/README.md
/content/data/diverse/aibelha/.gitattributes



In [37]:
import io, os, glob, json, numpy as np, pandas as pd, librosa, tensorflow as tf
from scipy.signal import resample
SR=16000; WIN=4.0; N_FFT=512
BANDS={'worker_hum':(200,300),'queen_pipe':(400,550),'whoop':(300,450),'worker_pipe':(100,250),'queen_harm':(500,2000),'sub':(0,100),'hiss':(3000,6000)}
FR=librosa.fft_frequencies(sr=SR,n_fft=N_FFT)
S=json.load(open('/content/data/run2/scaler.json')); mu,sd=np.array(S['mean'],np.float32),np.array(S['std'],np.float32)
def feats(y):
    G=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=160,win_length=400,window='hann')); P=G**2
    def be(lo,hi):
        m=(FR>=lo)&(FR<hi); return float(P[m].mean()) if m.any() else 0.0
    e={k:be(*v) for k,v in BANDS.items()}
    rms=float(np.sqrt((y**2).mean())+1e-12)
    cent=float((FR[:,None]*G).sum()/(G.sum()+1e-12)); zcr=float(((y[:-1]*y[1:])<0).mean())
    return np.array([20*np.log10(rms),*[np.log10(e[k]+1e-12) for k in ['worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss']],cent,np.log10((e['queen_pipe']+1e-12)/(e['worker_hum']+1e-12)),zcr],np.float32)
it=tf.lite.Interpreter(model_content=open('/content/data/run2/bee_dense_int8.tflite','rb').read()); it.allocate_tensors()
id_,od_=it.get_input_details()[0],it.get_output_details()[0]; qs,zp=id_['quantization']; os2,z2=od_['quantization']
def vote(F):
    outs=[]
    for x in F:
        q=np.clip(np.round(x/qs+zp),-128,127).astype(np.int8)
        it.set_tensor(id_['index'],q[None,:]); it.invoke()
        o=it.get_tensor(od_['index']).astype(np.float32)
        z=(o*os2+z2).ravel(); z=z-z.max(); e=np.exp(z); outs.append(e/e.sum())
    P=np.array(outs); f=float((P.argmax(1)==1).mean())
    return ('queenright' if f>0.5 else 'queenless'), round(f,3), len(F)
def windows(y):
    n=int(SR*WIN); return [(feats(y[s:s+n])-mu)/sd for s in range(0,len(y)-n+1,n)]
MAP={'queen_absent':'queenless','queen_present_original':'queenright'}
rep={}
# A) AI-Belha parquet
try:
    df=pd.read_parquet('/content/data/diverse/aibelha/data/train-00000-of-00001.parquet')
    print('aibelha cols:', list(df.columns))
    print('label counts:', df['user_obs_status_queen'].value_counts().to_dict())
    sub=df[df['user_obs_status_queen'].isin(MAP)]
    for _,row in sub.iterrows():
        au=row['audio']; raw=au['bytes'] if isinstance(au,dict) else au
        y,sr=librosa.load(io.BytesIO(raw),sr=None,mono=True)
        if sr!=SR: y=resample(y,int(len(y)*SR/sr)).astype(np.float32)
        pred,f,n=vote(np.array(windows(y)))
        true=MAP[row['user_obs_status_queen']]
        rep['aibelha:'+str(len(rep))]={'true':true,'pred':pred,'qr_frac':f,'n':n}
        print(f"aibelha true={true:10s} INT8={pred:10s} {'OK' if pred==true else 'WRONG'} qr={f} n={n}")
except Exception as e:
    print('aibelha FAIL:', type(e).__name__, e)
# B) loose files with filename labels
import re
for wav in sorted(glob.glob('/content/data/diverse/*.wav')+glob.glob('/content/data/diverse/*.mp3')):
    base=os.path.basename(wav)
    if 'gruber' in base: true='queenright' if 'without-queen' not in base and 'with-queen' in base else 'queenless'
    elif 'QueenBee' in base and 'NO_QueenBee' not in base and 'Missing' not in base: true='queenright'
    else: true='queenless'
    y,sr=librosa.load(wav,sr=None,mono=True)
    if sr!=SR: y=resample(y,int(len(y)*SR/sr)).astype(np.float32)
    pred,f,n=vote(np.array(windows(y)))
    rep['file:'+base]={'true':true,'pred':pred,'qr_frac':f,'n':n}
    print(f"{base[:44]:44s} true={true:10s} INT8={pred:10s} {'OK' if pred==true else 'WRONG'} qr={f} n={n}")
acc=sum(1 for v in rep.values() if v['true']==v['pred'])/max(1,len(rep))
print('DIVERSE total:', len(rep), 'acc:', round(acc,3))
json.dump(rep,open('/content/data/run2/diverse_test.json','w'),indent=2)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


aibelha cols: ['audio', 'key', 'user_obs_status_queen', 'pred_status_queen', 'pred_score', 'request_timestamp']
label counts: {'unknown ': 48, 'queen present or original queen': 15, 'queen not present': 15, 'queen present and newly accepted': 5, 'queen present and rejected': 3}
CJ001_-_Missing_Queen_-_Day_-__101.wav       true=queenless  INT8=queenless  OK qr=0.0 n=75
Hive1_12_06_2018_QueenBee_H1_audio___16_30_0 true=queenright INT8=queenright OK qr=0.537 n=149
Hive1_31_05_2018_NO_QueenBee_H1_audio___15_1 true=queenless  INT8=queenless  OK qr=0.041 n=148
Hive3_15_07_2017_NO_QueenBee_H3_audio___06_4 true=queenless  INT8=queenless  OK qr=0.324 n=145
colony-with-queen-gruber.mp3                 true=queenright INT8=queenless  WRONG qr=0.0 n=9
colony-without-queen-gruber.mp3              true=queenless  INT8=queenless  OK qr=0.222 n=9
DIVERSE total: 6 acc: 0.833


In [38]:
import io, json, numpy as np, pandas as pd, librosa, tensorflow as tf
from scipy.signal import resample
SR=16000; WIN=4.0; N_FFT=512
BANDS={'worker_hum':(200,300),'queen_pipe':(400,550),'whoop':(300,450),'worker_pipe':(100,250),'queen_harm':(500,2000),'sub':(0,100),'hiss':(3000,6000)}
FR=librosa.fft_frequencies(sr=SR,n_fft=N_FFT)
S=json.load(open('/content/data/run2/scaler.json')); mu,sd=np.array(S['mean'],np.float32),np.array(S['std'],np.float32)
def feats(y):
    G=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=160,win_length=400,window='hann')); P=G**2
    def be(lo,hi):
        m=(FR>=lo)&(FR<hi); return float(P[m].mean()) if m.any() else 0.0
    e={k:be(*v) for k,v in BANDS.items()}
    rms=float(np.sqrt((y**2).mean())+1e-12)
    cent=float((FR[:,None]*G).sum()/(G.sum()+1e-12)); zcr=float(((y[:-1]*y[1:])<0).mean())
    return np.array([20*np.log10(rms),*[np.log10(e[k]+1e-12) for k in ['worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss']],cent,np.log10((e['queen_pipe']+1e-12)/(e['worker_hum']+1e-12)),zcr],np.float32)
it=tf.lite.Interpreter(model_content=open('/content/data/run2/bee_dense_int8.tflite','rb').read()); it.allocate_tensors()
id_,od_=it.get_input_details()[0],it.get_output_details()[0]; qs,zp=id_['quantization']; os2,z2=od_['quantization']
def vote(F):
    outs=[]
    for x in F:
        q=np.clip(np.round(x/qs+zp),-128,127).astype(np.int8)
        it.set_tensor(id_['index'],q[None,:]); it.invoke()
        o=it.get_tensor(od_['index']).astype(np.float32)
        z=(o*os2+z2).ravel(); z=z-z.max(); e=np.exp(z); outs.append(e/e.sum())
    P=np.array(outs); f=float((P.argmax(1)==1).mean())
    return ('queenright' if f>0.5 else 'queenless'), round(f,3), len(F)
MAP={'queen not present':'queenless','queen present or original queen':'queenright'}
df=pd.read_parquet('/content/data/diverse/aibelha/data/train-00000-of-00001.parquet')
rep=json.load(open('/content/data/run2/diverse_test.json'))
n_ok=n_tot=0
for i,row in df[df['user_obs_status_queen'].isin(MAP)].iterrows():
    au=row['audio']; raw=au['bytes'] if isinstance(au,dict) else au
    y,sr=librosa.load(io.BytesIO(raw),sr=None,mono=True)
    if sr!=SR: y=resample(y,int(len(y)*SR/sr)).astype(np.float32)
    n=int(SR*WIN); F=np.array([(feats(y[s:s+n])-mu)/sd for s in range(0,len(y)-n+1,n)])
    pred,f,nw=vote(F); true=MAP[row['user_obs_status_queen']]
    rep['aibelha-'+str(i)]={'true':true,'pred':pred,'qr_frac':f,'n':nw}
    n_tot+=1; n_ok+= (pred==true)
    if i<3 or pred!=true: print(f"aibelha#{i} true={true:10s} INT8={pred:10s} {'OK' if pred==true else 'WRONG'} qr={f} n={nw}")
print(f'aibelha clean-label: {n_ok}/{n_tot}')
acc=sum(1 for v in rep.values() if v['true']==v['pred'])/len(rep)
print('GRAND total files:',len(rep),'acc:',round(acc,3))
json.dump(rep,open('/content/data/run2/diverse_test.json','w'),indent=2)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


aibelha#1 true=queenright INT8=queenright OK qr=1.0 n=15
aibelha#2 true=queenless  INT8=queenless  OK qr=0.357 n=14
aibelha#15 true=queenless  INT8=queenright WRONG qr=0.714 n=14
aibelha#18 true=queenless  INT8=queenright WRONG qr=0.933 n=15
aibelha#29 true=queenright INT8=queenless  WRONG qr=0.077 n=13
aibelha#30 true=queenless  INT8=queenright WRONG qr=0.571 n=14
aibelha#37 true=queenright INT8=queenless  WRONG qr=0.067 n=15
aibelha#38 true=queenless  INT8=queenright WRONG qr=1.0 n=14
aibelha#42 true=queenless  INT8=queenright WRONG qr=0.933 n=15
aibelha#45 true=queenright INT8=queenless  WRONG qr=0.133 n=15
aibelha#48 true=queenright INT8=queenless  WRONG qr=0.0 n=15
aibelha#56 true=queenless  INT8=queenright WRONG qr=0.933 n=15
aibelha#57 true=queenright INT8=queenless  WRONG qr=0.067 n=15
aibelha#66 true=queenless  INT8=queenright WRONG qr=1.0 n=15
aibelha#72 true=queenless  INT8=queenright WRONG qr=0.571 n=14
aibelha#76 true=queenless  INT8=queenright WRONG qr=0.6 n=15
aibelha cl

In [39]:
from google.colab import files
files.download('/content/data/run2/diverse_test.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
import io, os, glob, numpy as np, pandas as pd, librosa
from scipy.signal import resample
SR=16000; WIN=4.0; N_FFT=512
BANDS={'worker_hum':(200,300),'queen_pipe':(400,550),'whoop':(300,450),'worker_pipe':(100,250),'queen_harm':(500,2000),'sub':(0,100),'hiss':(3000,6000)}
FR=librosa.fft_frequencies(sr=SR,n_fft=N_FFT)
def feats(y):
    G=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=160,win_length=400,window='hann')); P=G**2
    def be(lo,hi):
        m=(FR>=lo)&(FR<hi); return float(P[m].mean()) if m.any() else 0.0
    e={k:be(*v) for k,v in BANDS.items()}
    rms=float(np.sqrt((y**2).mean())+1e-12)
    cent=float((FR[:,None]*G).sum()/(G.sum()+1e-12)); zcr=float(((y[:-1]*y[1:])<0).mean())
    return np.array([20*np.log10(rms),*[np.log10(e[k]+1e-12) for k in ['worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss']],cent,np.log10((e['queen_pipe']+1e-12)/(e['worker_hum']+1e-12)),zcr],np.float32)
def segs(lab_path, dur):
    out=[]
    try:
        for ln in open(lab_path).read().strip().split(chr(10))[1:]:
            p=ln.split()
            if len(p)>=3: out.append((p[2],max(0.0,float(p[0])),min(dur,float(p[1]))))
    except Exception: return [('bee',0.0,dur)]
    return out or [('bee',0.0,dur)]
Xv,Y,S,H=[],[],[],[]
for wav in sorted(glob.glob('/content/data/run2_wav/*.wav')):
    base=os.path.basename(wav)
    q='queenright' if ('QueenBee' in base and 'NO_QueenBee' not in base and 'Missing' not in base) else 'queenless'
    hive='hive1' if 'Hive1' in base else ('hive3' if 'Hive3' in base else 'cj001')
    y,sr=librosa.load(wav,sr=None,mono=True)
    if sr!=SR: y=resample(y,int(len(y)*SR/sr)).astype(np.float32)
    dur=len(y)/SR; n=int(SR*WIN); nw=0
    for (c,a,b) in segs(wav[:-4]+'.lab', dur):
        lab=q if c=='bee' else 'nobee'
        for st in range(int(a*SR), int(min(b,dur)*SR)-n+1, n):
            Xv.append(feats(y[st:st+n])); Y.append(lab); S.append('nuhive'); H.append(hive); nw+=1
    print(base[:42], 'wins='+str(nw), flush=True)
MAP={'queen not present':'queenless','queen present or original queen':'queenright'}
df=pd.read_parquet('/content/data/diverse/aibelha/data/train-00000-of-00001.parquet')
for i,row in df[df['user_obs_status_queen'].isin(MAP)].iterrows():
    au=row['audio']; raw=au['bytes'] if isinstance(au,dict) else au
    y,sr=librosa.load(io.BytesIO(raw),sr=None,mono=True)
    if sr!=SR: y=resample(y,int(len(y)*SR/sr)).astype(np.float32)
    n=int(SR*WIN)
    for st in range(0,len(y)-n+1,n):
        Xv.append(feats(y[st:st+n])); Y.append(MAP[row['user_obs_status_queen']]); S.append('aibelha'); H.append('pt'+str(int(i)%6))
Xv,Y,S,H=map(np.array,[Xv,Y,S,H])
print('TOTAL',Xv.shape,dict(zip(*np.unique(Y,return_counts=True))),'| src:',dict(zip(*np.unique(S,return_counts=True))))
np.savez('/content/data/run3_feat.npz',X_vec=Xv,y=Y,src=S,hives=H)
print('saved run3_feat.npz')


CJ001_-_Missing_Queen_-_Day_-__102.wav wins=65
Hive1_12_06_2018_QueenBee_H1_audio___16_50 wins=142
Hive1_12_06_2018_QueenBee_H1_audio___17_00 wins=142
Hive1_31_05_2018_NO_QueenBee_H1_audio___15 wins=134
Hive3_12_07_2017_NO_QueenBee_H3_audio___17 wins=139
Hive3_20_07_2017_QueenBee_H3_audio___06_10 wins=130
TOTAL (1195, 11) {np.str_('nobee'): np.int64(204), np.str_('queenless'): np.int64(467), np.str_('queenright'): np.int64(524)} | src: {np.str_('aibelha'): np.int64(443), np.str_('nuhive'): np.int64(752)}
saved run3_feat.npz


In [41]:
import json, numpy as np, tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix
tf.random.set_seed(7); np.random.seed(7)
D=np.load('/content/data/run3_feat.npz'); Xv,Y,S=D['X_vec'].astype(np.float32),D['y'],D['src']
classes=['nobee','queenless','queenright']; yi=np.array([classes.index(c) for c in Y])
print('n=',len(Y),{c:int((Y==c).sum()) for c in classes})
sss=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=7)
tr,te=next(sss.split(Xv,yi))
mu,sd=Xv[tr].mean(0),Xv[tr].std(0)+1e-9; Z=((Xv-mu)/sd).astype(np.float32)
json.dump({'mean':mu.tolist(),'std':sd.tolist()},open('/content/data/run2/scaler3.json','w'))
def net3():
    m=keras.Sequential([keras.Input((11,)),keras.layers.Dense(64,activation='relu'),keras.layers.Dense(32,activation='relu'),keras.layers.Dense(3,activation='softmax')])
    m.compile(optimizer=keras.optimizers.Adam(3e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
net=net3()
cb=[keras.callbacks.EarlyStopping(patience=25,restore_best_weights=True,monitor='val_accuracy')]
h=net.fit(Z[tr],yi[tr],validation_data=(Z[te],yi[te]),epochs=300,batch_size=64,callbacks=cb,verbose=0)
print('epochs:',len(h.history['loss']),'best val:',round(max(h.history['val_accuracy']),3))
pc=net.predict(Z[te],verbose=0).argmax(1)
print('holdout acc:',round(float(accuracy_score(yi[te],pc)),3)); print(confusion_matrix(yi[te],pc))
for src in ['nuhive','aibelha']:
    a,b=S!=src,S==src
    if len(set(yi[a]))<3: continue
    m2=net3(); m2.fit(Z[a],yi[a],epochs=150,batch_size=64,verbose=0)
    q=float(accuracy_score(yi[b],m2.predict(Z[b],verbose=0).argmax(1)))
    print(f'leave-{src}-out: {q:.3f} (train_n={a.sum()}, test_n={b.sum()})')
net.save('/content/data/run2/keras_3class.keras')
print('params:',net.count_params())


n= 1195 {'nobee': 204, 'queenless': 467, 'queenright': 524}
epochs: 50 best val: 0.799
holdout acc: 0.799
[[23  8 10]
 [ 7 80  6]
 [ 4 13 88]]
leave-aibelha-out: 0.501 (train_n=752, test_n=443)
params: 2947


In [42]:
import numpy as np, tensorflow as tf
import json
D = np.load('/content/data/run3_feat.npz')
X = D['X_vec'].astype(np.float32)
S = json.load(open('/content/data/run2/scaler3.json'))
mu, sd = np.array(S['mean'], np.float32), np.array(S['std'], np.float32)
Z = ((X - mu) / sd).astype(np.float32)
net = tf.keras.models.load_model('/content/data/run2/keras_3class.keras')
def rep():
    for i in range(0, len(Z), 5):
        yield [Z[i:i+1]]
conv = tf.lite.TFLiteConverter.from_keras_model(net)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl = conv.convert()
open('/content/data/run2/bee_3class_int8.tflite', 'wb').write(tfl)
print('int8 bytes:', len(tfl))
ref = net.predict(Z[:200], verbose=0).argmax(1)
it = tf.lite.Interpreter(model_content=tfl)
it.allocate_tensors()
a = it.get_input_details()[0]
b = it.get_output_details()[0]
qs, zp = a['quantization']
s2, z2 = b['quantization']
agree = 0
for i, x in enumerate(Z[:200]):
    q8 = np.clip(np.round(x / qs + zp), -128, 127).astype(np.int8)
    it.set_tensor(a['index'], q8[None, :])
    it.invoke()
    o = it.get_tensor(b['index']).astype(np.float32)
    z = (o * s2 + z2).ravel()
    z = z - z.max()
    e = np.exp(z)
    p = e / e.sum()
    if int(p.argmax()) == int(ref[i]):
        agree = agree + 1
print('keras-vs-int8 agreement:', agree, '/ 200')

Saved artifact at '/tmp/tmppliwawc7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 11), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  137174814665680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137174814668176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137174814664912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137174814669520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137174814668944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137174814669904: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


int8 bytes: 8032
keras-vs-int8 agreement: 199 / 200


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [43]:
import io, os, glob, json, numpy as np, pandas as pd, librosa, tensorflow as tf
from scipy.signal import resample
SR=16000; WIN=4.0; N_FFT=512
BANDS={'worker_hum':(200,300),'queen_pipe':(400,550),'whoop':(300,450),'worker_pipe':(100,250),'queen_harm':(500,2000),'sub':(0,100),'hiss':(3000,6000)}
FR=librosa.fft_frequencies(sr=SR,n_fft=N_FFT)
S=json.load(open('/content/data/run2/scaler3.json')); mu,sd=np.array(S['mean'],np.float32),np.array(S['std'],np.float32)
CLS=['nobee','queenless','queenright']
def feats(y):
    G=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=160,win_length=400,window='hann')); P=G**2
    def be(lo,hi):
        m=(FR>=lo)&(FR<hi); return float(P[m].mean()) if m.any() else 0.0
    e={k:be(*v) for k,v in BANDS.items()}
    rms=float(np.sqrt((y**2).mean())+1e-12)
    cent=float((FR[:,None]*G).sum()/(G.sum()+1e-12)); zcr=float(((y[:-1]*y[1:])<0).mean())
    return np.array([20*np.log10(rms),*[np.log10(e[k]+1e-12) for k in ['worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss']],cent,np.log10((e['queen_pipe']+1e-12)/(e['worker_hum']+1e-12)),zcr],np.float32)
it=tf.lite.Interpreter(model_content=open('/content/data/run2/bee_3class_int8.tflite','rb').read()); it.allocate_tensors()
a=it.get_input_details()[0]; b=it.get_output_details()[0]; qs,zp=a['quantization']; s2,z2=b['quantization']
def runfile(y, tag, true):
    n=int(SR*WIN); F=np.array([(feats(y[s:s+n])-mu)/sd for s in range(0,len(y)-n+1,n)])
    outs=[]
    for x in F:
        q8=np.clip(np.round(x/qs+zp),-128,127).astype(np.int8)
        it.set_tensor(a['index'],q8[None,:]); it.invoke()
        o=it.get_tensor(b['index']).astype(np.float32)
        z=(o*s2+z2).ravel(); z=z-z.max(); e=np.exp(z); outs.append(e/e.sum())
    P=np.array(outs); v=P.argmax(1); fr=[round(float((v==i).mean()),3) for i in range(3)]
    pred=CLS[int(np.argmax(np.bincount(v)))]
    print(tag + ' true='+true+' INT8='+pred+('OK' if pred==true else 'WRONG')+' fr='+str(fr)+' n='+str(len(F)))
    return {'true':true,'pred':pred,'frac':fr,'n':len(F)}
def loadwav(p):
    y,sr=librosa.load(p,sr=None,mono=True)
    return (resample(y,int(len(y)*SR/sr)).astype(np.float32) if sr!=SR else y)
rep={}
MAP={'queen not present':'queenless','queen present or original queen':'queenright'}
df=pd.read_parquet('/content/data/diverse/aibelha/data/train-00000-of-00001.parquet')
sub=df[df['user_obs_status_queen'].isin(MAP)]
ai_ok=0
for i,row in sub.iterrows():
    au=row['audio']; raw=au['bytes'] if isinstance(au,dict) else au
    y,sr=librosa.load(io.BytesIO(raw),sr=None,mono=True)
    y=(resample(y,int(len(y)*SR/sr)).astype(np.float32) if sr!=SR else y)
    r=runfile(y,'aibelha-'+str(i),MAP[row['user_obs_status_queen']]); rep['aibelha-'+str(i)]=r; ai_ok+=(r['pred']==r['true'])
print('aibelha file-acc:',ai_ok,'/',len(sub))
import re
for wav in sorted(glob.glob('/content/data/diverse/*.wav')+glob.glob('/content/data/diverse/*.mp3')):
    base=os.path.basename(wav)
    true=('queenright' if ('with-queen' in base) else ('queenless' if ('without-queen' in base or 'NO_QueenBee' in base or 'Missing' in base) else ('queenright' if 'QueenBee' in base else 'nobee')))
    rep['file:'+base]=runfile(loadwav(wav),base[:40],true)
acc=sum(1 for v in rep.values() if v['true']==v['pred'])/len(rep)
print('GRAND files:',len(rep),'acc:',round(acc,3))
json.dump(rep,open('/content/data/run2/diverse3_test.json','w'),indent=2)

aibelha-1 true=queenright INT8=queenlessWRONG fr=[0.0, 0.533, 0.467] n=15
aibelha-2 true=queenless INT8=queenlessOK fr=[0.0, 0.929, 0.071] n=14
aibelha-4 true=queenless INT8=queenlessOK fr=[0.0, 0.867, 0.133] n=15
aibelha-10 true=queenright INT8=queenrightOK fr=[0.0, 0.333, 0.667] n=15
aibelha-15 true=queenless INT8=queenlessOK fr=[0.0, 0.929, 0.071] n=14
aibelha-18 true=queenless INT8=queenlessOK fr=[0.0, 0.933, 0.067] n=15
aibelha-19 true=queenright INT8=queenrightOK fr=[0.133, 0.0, 0.867] n=15
aibelha-23 true=queenless INT8=queenlessOK fr=[0.0, 1.0, 0.0] n=15
aibelha-27 true=queenright INT8=queenrightOK fr=[0.0, 0.333, 0.667] n=15
aibelha-29 true=queenright INT8=queenlessWRONG fr=[0.0, 0.538, 0.462] n=13
aibelha-30 true=queenless INT8=queenlessOK fr=[0.0, 1.0, 0.0] n=14
aibelha-33 true=queenless INT8=queenlessOK fr=[0.0, 1.0, 0.0] n=15
aibelha-35 true=queenright INT8=queenrightOK fr=[0.0, 0.0, 1.0] n=15
aibelha-37 true=queenright INT8=queenrightOK fr=[0.0, 0.267, 0.733] n=15
aibelha

In [44]:
from google.colab import files
for f in ['diverse3_test.json', 'bee_3class_int8.tflite', 'scaler3.json', 'keras_3class.keras']:
    files.download('/content/data/run2/' + f)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
import io, json, numpy as np, pandas as pd, librosa, tensorflow as tf
from scipy.signal import resample
SR=16000; WIN=4.0; N_FFT=512
BANDS={'worker_hum':(200,300),'queen_pipe':(400,550),'whoop':(300,450),'worker_pipe':(100,250),'queen_harm':(500,2000),'sub':(0,100),'hiss':(3000,6000)}
FR=librosa.fft_frequencies(sr=SR,n_fft=N_FFT)
S=json.load(open('/content/data/run2/scaler3.json')); mu,sd=np.array(S['mean'],np.float32),np.array(S['std'],np.float32)
CLS=['nobee','queenless','queenright']
def feats(y):
    G=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=160,win_length=400,window='hann')); P=G**2
    def be(lo,hi):
        m=(FR>=lo)&(FR<hi); return float(P[m].mean()) if m.any() else 0.0
    e={k:be(*v) for k,v in BANDS.items()}
    rms=float(np.sqrt((y**2).mean())+1e-12)
    cent=float((FR[:,None]*G).sum()/(G.sum()+1e-12)); zcr=float(((y[:-1]*y[1:])<0).mean())
    return np.array([20*np.log10(rms),*[np.log10(e[k]+1e-12) for k in ['worker_hum','queen_pipe','whoop','worker_pipe','queen_harm','sub','hiss']],cent,np.log10((e['queen_pipe']+1e-12)/(e['worker_hum']+1e-12)),zcr],np.float32)
it=tf.lite.Interpreter(model_content=open('/content/data/run2/bee_3class_int8.tflite','rb').read()); it.allocate_tensors()
a=it.get_input_details()[0]; b=it.get_output_details()[0]; qs,zp=a['quantization']; s2,z2=b['quantization']
KEEP={'queen present and newly accepted','queen present and rejected'}
df=pd.read_parquet('/content/data/diverse/aibelha/data/train-00000-of-00001.parquet')
sub=df[df['user_obs_status_queen'].isin(KEEP)]
print('transition clips:', len(sub))
for i,row in sub.iterrows():
    au=row['audio']; raw=au['bytes'] if isinstance(au,dict) else au
    y,sr=librosa.load(io.BytesIO(raw),sr=None,mono=True)
    y=(resample(y,int(len(y)*SR/sr)).astype(np.float32) if sr!=SR else y)
    n=int(SR*WIN); F=np.array([(feats(y[s:s+n])-mu)/sd for s in range(0,len(y)-n+1,n)])
    outs=[]
    for x in F:
        q8=np.clip(np.round(x/qs+zp),-128,127).astype(np.int8)
        it.set_tensor(a['index'],q8[None,:]); it.invoke()
        o=it.get_tensor(b['index']).astype(np.float32)
        z=(o*s2+z2).ravel(); z=z-z.max(); e=np.exp(z); outs.append(e/e.sum())
    P=np.array(outs); fr=[round(float((P.argmax(1)==j)).mean(),3) if False else round(float((P.argmax(1)==j).mean()),3) for j in range(3)]
    print(row['user_obs_status_queen'][:28] + ' -> ' + str(fr) + ' n=' + str(len(F)))

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


transition clips: 8
queen present and newly acce -> [0.0, 1.0, 0.0] n=15
queen present and newly acce -> [0.0, 0.667, 0.333] n=15
queen present and rejected -> [0.0, 0.933, 0.067] n=15
queen present and newly acce -> [0.0, 0.6, 0.4] n=15
queen present and rejected -> [0.0, 0.8, 0.2] n=15
queen present and newly acce -> [0.0, 0.933, 0.067] n=15
queen present and rejected -> [0.0, 0.867, 0.133] n=15
queen present and newly acce -> [0.0, 0.867, 0.133] n=15


In [46]:
import os
print('run3_feat.npz bytes:', os.path.getsize('/content/data/run3_feat.npz'))
from google.colab import files
files.download('/content/data/run3_feat.npz')


run3_feat.npz bytes: 158718


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
import subprocess
print(subprocess.run(['df','-h','/content'], capture_output=True, text=True).stdout)


Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   22G   87G  21% /



In [48]:
import json, urllib.request
for rec in ['321345', '2667806'] :
    r = json.load(urllib.request.urlopen('https://zenodo.org/api/records/'+rec, timeout=60))
    fs = r.get('files', [])
    tot = sum(f.get('size',0) for f in fs)
    print(rec, len(fs), 'files,', round(tot/1e9,2), 'GB')
    keys = [f['key'] for f in fs]
    uw = [k for k in keys if k.endswith('.wav')]
    print('  wavs:', len(uw))
    print('  sample:', keys[:6])

321345 1 files, 0.5 GB
  wavs: 0
  sample: ['Current Audio.zip']
2667806 6 files, 23.1 GB
  wavs: 0
  sample: ['state_labels.csv', 'Hive3_14_07_2017.rar', 'Hive1_31_05_2018.rar', 'Hive3_28_07_2017.rar', 'Hive1_12_06_2018.rar', 'Documentation.pdf']


In [ ]:
import os, subprocess, urllib.request, urllib.parse
OD='/content/data/bigpull'
os.makedirs(OD, exist_ok=True)
JOBS=['321345|Current Audio.zip','2667806|state_labels.csv','2667806|Hive1_31_05_2018.rar','2667806|Hive1_12_06_2018.rar','2667806|Hive3_14_07_2017.rar','2667806|Hive3_28_07_2017.rar']
BASE='https://zenodo.org/records/'
for job in JOBS:
    rec, key = job.split('|')
    dst = os.path.join(OD, key)
    if os.path.exists(dst) and os.path.getsize(dst) > 5000000:
        print('have '+key); continue
    print('dl '+key)
    urllib.request.urlretrieve(BASE+rec+'/files/'+urllib.parse.quote(key)+'?download=1', dst)
    print('  MB:', os.path.getsize(dst)//1000000)
print('--- dir ---')
print(subprocess.run(['ls','-la',OD],capture_output=True,text=True).stdout)

dl Current Audio.zip
  MB: 504
dl state_labels.csv
  MB: 0
dl Hive1_31_05_2018.rar
  MB: 5084
dl Hive1_12_06_2018.rar
  MB: 5360
dl Hive3_14_07_2017.rar
